In [5]:
def run_stage1_validation(data_path):
    print(f"\n==========================================")
    print(f"STAGE 1 VALIDATION (Great Expectations) ON: {data_path}")
    print(f"==========================================")
    
    # 1. Load dataset & filter relevant transaction types
    df_raw = pd.read_csv(data_path)
    df_filtered = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # Measure validation timing (T_valid)
    t_start_valid = time.time()
    
    # 2. Setup Great Expectations Validator for modern GE versions
    suite = ExpectationSuite(name="paysim_ingestion_suite")
    engine = PandasExecutionEngine()
    validator = Validator(
        execution_engine=engine,
        expectation_suite=suite,
        in_memory_dataset=df_filtered
    )
    
    # 3. Define Declarative Expectations (Contracts)
    target_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
    
    # Rule 1: Columns must not contain null/missing values
    for col in target_cols:
        validator.expect_column_values_to_not_be_null(column=col)
        
    # Rule 2: Transaction amounts must be non-negative
    validator.expect_column_values_to_be_between(column='amount', min_value=0)
    
    # Rule 3: Balance columns must exist in schema
    validator.expect_column_to_exist(column='oldbalanceOrg')
    validator.expect_column_to_exist(column='newbalanceOrig')
    
    # 4. Validate Micro-batch Data
    validation_results = validator.validate()
    t_end_valid = time.time()
    
    # 5. Extract Metrics
    t_valid_duration = t_end_valid - t_start_valid
    success = validation_results["success"]
    stats = validation_results["statistics"]
    
    print(f"Validation Status: {'SUCCESS (Pristine Data)' if success else 'FAILED (Anomalies Detected)'}")
    print(f"Validation Time (T_valid): {t_valid_duration:.4f} seconds")
    print(f"Evaluated Rules: {stats['evaluated_expectations']}")
    print(f"Passed Rules: {stats['successful_expectations']}")
    print(f"Failed Rules: {stats['unsuccessful_expectations']}")
    
    return validator, validation_results, t_valid_duration

# Execute validation on pristine data
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('Dataset.csv')


STAGE 1 VALIDATION (Great Expectations) ON: Dataset.csv


TypeError: active_batch cannot be None

In [8]:
import pandas as pd
import great_expectations as ge
import time

def run_stage1_validation(data_path):
    df_raw = pd.read_csv(data_path)
    df_filtered = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # Simple approach using from_pandas (requires GE 0.15.x or lower)
    # But this might not work with your current version
    
    # Alternative: Just use pandas for validation
    t_start_valid = time.time()
    
    # Manual validation
    validation_results = {
        'success': True,
        'errors': [],
        'warnings': []
    }
    
    # Check for null values
    if df_filtered['type'].isnull().any():
        validation_results['success'] = False
        validation_results['errors'].append("Null values found in 'type' column")
    
    # Check amount column
    if df_filtered['amount'].isnull().any():
        validation_results['success'] = False
        validation_results['errors'].append("Null values found in 'amount' column")
    
    if (df_filtered['amount'] < 0).any():
        validation_results['success'] = False
        validation_results['errors'].append("Negative amounts found")
    
    # Check for duplicates
    if df_filtered.duplicated().any():
        validation_results['warnings'].append("Duplicate rows found")
    
    t_valid = time.time() - t_start_valid
    
    return df_filtered, validation_results, t_valid

# Execute
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('Dataset.csv')
    print(f"Validation success: {results['success']}")
    print(f"Errors: {results['errors']}")
    print(f"Time taken: {t_valid:.4f} seconds")

Validation success: True
Errors: []
Time taken: 7.4521 seconds


In [9]:
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('paysim_noise_5.csv')
    print(f"Validation success: {results['success']}")
    print(f"Errors: {results['errors']}")
    print(f"Time taken: {t_valid:.4f} seconds")

Validation success: False
Errors: ["Null values found in 'amount' column"]
Time taken: 8.1172 seconds


In [10]:
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('paysim_noise_10.csv')
    print(f"Validation success: {results['success']}")
    print(f"Errors: {results['errors']}")
    print(f"Time taken: {t_valid:.4f} seconds")

Validation success: False
Errors: ["Null values found in 'amount' column"]
Time taken: 9.3868 seconds


In [11]:
if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('paysim_noise_20.csv')
    print(f"Validation success: {results['success']}")
    print(f"Errors: {results['errors']}")
    print(f"Time taken: {t_valid:.4f} seconds")

Validation success: False
Errors: ["Null values found in 'amount' column"]
Time taken: 9.2887 seconds
